# M6 — DTM artifact detector (Marswalk)

Trains a small CNN to flag **DEM/DTM artifacts** — box-steps and CCD-seam lines — from hillshade. Labels are made by injecting artifacts of known type into clean synthetic DEMs (honest weak supervision); you then run the detector over real HiRISE DTM hillshades to flag suspect tiles.

**How to run:** Runtime → GPU (optional), **Run all**, then send back **`dtm_artifacts_report.json`** and **`dtm_artifacts_samples.png`**.

In [ ]:
!pip install -q "git+https://github.com/MONSTER4REX/Martian-Map.git"
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', dev,
      '|', torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU')

## Build labelled patches, train, evaluate on unseen terrain

Train and test terrains are disjoint, so the score reflects generalization, not memorized texture.

In [ ]:
import json, time
import numpy as np
from marswalk.ml.dtm_artifacts import make_dataset, train_cnn, evaluate

Xtr, ytr = make_dataset(range(60), per_terrain=16, step_frac=0.25)
Xte, yte = make_dataset(range(1000, 1012), per_terrain=16, step_frac=0.25)
print(f'train {len(Xtr)} patches | test {len(Xte)} (disjoint terrains)')
t0 = time.time()
model = train_cnn(Xtr, ytr, epochs=30, device=dev)
rep = evaluate(model, Xte, yte)
secs = time.time() - t0
print(rep)

In [ ]:
report = {**rep.as_dict(), 'device': (torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU'),
          'epochs': 30, 'seconds': round(secs, 1), 'final_loss': round(model.history[-1], 5)}
with open('dtm_artifacts_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pred = model.predict(Xte)
lab = {0: 'clean', 1: 'artifact'}
idx = np.random.default_rng(1).choice(len(Xte), size=8, replace=False)
fig, ax = plt.subplots(2, 4, figsize=(11, 6))
for a, i in zip(ax.ravel(), idx):
    a.imshow(Xte[i, 0], cmap='gray'); a.set_xticks([]); a.set_yticks([])
    ok = pred[i] == yte[i]
    a.set_title(f"GT {lab[yte[i]]} / pred {lab[pred[i]]}",
                color=('green' if ok else 'red'), fontsize=9)
fig.suptitle(f"DTM artifact detector — test acc {rep.accuracy*100:.1f}%, F1 {rep.f1:.2f}")
fig.tight_layout(); fig.savefig('dtm_artifacts_samples.png', dpi=120)
import os; print('SAVED (download these):', os.path.abspath('dtm_artifacts_samples.png'), '|', os.path.abspath('dtm_artifacts_report.json'))

## Send back

Download **`dtm_artifacts_report.json`** and **`dtm_artifacts_samples.png`** and send them to the lead, plus your GPU name. Thanks!